<h1>RAZ Systems </h1>

Problem Statement

You are tasked with building a simple banking system simulator using Python and SQLite. The system should allow users to store and retrieve bank account information such as customer name and account balance.

In addition to basic database operations, you will design a smart assistant agent that can respond to user queries about bank accounts in natural language.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### First concept: Select Which Model to use

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

### Second concept: Construct the Message

In [3]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="What is the balance for account ACC1003?", source="user")
message

TextMessage(id='c99eadf4-6c71-4227-803e-647afb579ef3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 14, 9, 51, 19, 523068, tzinfo=datetime.timezone.utc), content='What is the balance for account ACC1003?', type='TextMessage')

### Third concept: Build an Agent

In [4]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful banking assistant. "
        "You provide short, professional answers about customer bank account balances."
    ),
    model_client_stream=True
)

### Put it all together with on_messages

In [5]:
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'I’m sorry, but I cannot access specific account balances or confidential information. Please check your online banking portal or contact customer service for assistance with your account.'

### Let's make a local database of ticket prices

In [6]:
import os
import sqlite3

# Delete existing database file if it exists
if os.path.exists("bank.db"):
    os.remove("bank.db")

# Create the database and the table
conn = sqlite3.connect("bank.db")

c = conn.cursor()

c.execute("""
CREATE TABLE bank_accounts (
    account_number TEXT PRIMARY KEY,
    customer_name TEXT,
    balance REAL
)
""")

conn.commit()
conn.close()

In [16]:
# Populate our database
def save_account_balance(account_number, customer_name, balance):
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()

    c.execute("""
    REPLACE INTO bank_accounts
    (account_number, customer_name, balance)
    VALUES (?, ?, ?)
    """, (account_number, customer_name, balance))

    conn.commit()
    conn.close()


# Sample bank accounts
save_account_balance("ACC1001", "Ajaz Pasha", 15000.75)
save_account_balance("ACC1002", "Aiza Khan", 24500.00)
save_account_balance("ACC1003", "Numan Khan", 8750.50)
save_account_balance("ACC1004", "Mohammed", 32000.25)
save_account_balance("ACC1005", "N Jahan", 12600.00)

In [17]:
def get_account_balance(account_number: str) ->str:
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()

    c.execute("""
    SELECT customer_name, balance
    FROM bank_accounts
    WHERE account_number = ?
    """, (account_number,))

    result = c.fetchone()

    conn.close()

    if result:
        customer_name, balance = result
        return f"Customer: {customer_name}, Balance: ${balance}"
    else:
        return "Account not found"



In [18]:

# Example usage
print(get_account_balance("ACC1003"))

Customer: Numan Khan, Balance: $8750.5


In [13]:
from autogen_agentchat.agents import AssistantAgent

smart_bank_agent = AssistantAgent(
    name="smart_bank_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful banking assistant. "
        "You provide short, professional answers about customer bank account balances."
    ),
    model_client_stream=True,
    tools=[get_account_balance],
    reflect_on_tool_use=True
)

In [14]:
response = await smart_bank_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_KnBU57u2GABCM0Orhq7BaauZ', arguments='{"account_number":"ACC1003"}', name='get_account_balance')]
[FunctionExecutionResult(content='Customer: Michael Brown, Balance: $8750.5', name='get_account_balance', call_id='call_KnBU57u2GABCM0Orhq7BaauZ', is_error=False)]


'The balance for account ACC1003 is $8,750.50.'

In [15]:
message

TextMessage(id='c99eadf4-6c71-4227-803e-647afb579ef3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 14, 9, 51, 19, 523068, tzinfo=datetime.timezone.utc), content='What is the balance for account ACC1003?', type='TextMessage')